# 01. Fundamentos de Agentes Inteligentes

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** Conocimientos básicos de Python y programación orientada a objetos

## 🎯 Objetivos de Aprendizaje
Al finalizar este notebook, podrás:
- Definir qué es un agente inteligente y sus componentes fundamentales
- Comprender el ciclo percepción-acción que gobierna el comportamiento de un agente
- Implementar un ambiente simple (GridWorld) y un agente que interactúa con él
- Entender el concepto de recompensas y cómo guían el aprendizaje
- Distinguir entre agentes reactivos y agentes que aprenden de la experiencia

## 📚 Motivación

### El Problema del Aprendizaje Autónomo

Imagina que quieres programar un robot para que navegue en un almacén y recoja paquetes. A diferencia de un programa tradicional donde especificas cada paso ("avanza 3 metros, gira 90 grados"), te enfrentas a varios desafíos:

1. **Incertidumbre**: El robot no sabe exactamente dónde están los paquetes
2. **Exploración**: Debe descubrir el ambiente por sí mismo
3. **Consecuencias diferidas**: Una acción ahora puede afectar el éxito futuro
4. **Objetivos**: Solo sabes que quieres "recoger paquetes eficientemente", pero ¿cómo se traduce eso en instrucciones?

Este es el dominio del **Reinforcement Learning** (Aprendizaje por Refuerzo). En lugar de decirle al agente QUÉ hacer, le dices QUÉ lograr, y él aprende CÓMO hacerlo mediante prueba y error.

### ¿Por qué es diferente?

- **Supervised Learning**: "Esta imagen es un gato" → Aprendizaje con etiquetas directas
- **Unsupervised Learning**: "Encuentra patrones en estos datos" → Aprendizaje sin etiquetas
- **Reinforcement Learning**: "Maximiza esta recompensa" → Aprendizaje mediante interacción

### Pregunta Guía
**¿Cómo puede un agente aprender a comportarse óptimamente en un ambiente desconocido usando solo señales de recompensa?**

Al final de este notebook, tendrás la base conceptual para responder esta pregunta.

In [ ]:
# Instalación de dependencias
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
import time
from typing import Tuple, List, Optional

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Semilla para reproducibilidad
np.random.seed(42)

print("✅ Librerías importadas correctamente")

## 🎨 Intuición Visual

### El Ciclo Percepción-Acción

Todo agente inteligente funciona mediante un ciclo fundamental:

```
Ambiente → [Observación] → Agente → [Acción] → Ambiente
    ↑                                             ↓
    └────────────── [Recompensa] ←───────────────┘
```

Este ciclo se repite continuamente. Veamos un ejemplo concreto:

In [ ]:
# Visualización del ciclo percepción-acción
fig = go.Figure()

# Crear diagrama de flujo
fig.add_trace(go.Scatter(
    x=[1, 2, 3, 2, 1],
    y=[2, 3, 2, 1, 2],
    mode='lines+markers+text',
    text=['Ambiente', 'Agente', 'Ambiente', 'Agente', 'Ambiente'],
    textposition='top center',
    marker=dict(size=20, color=['lightblue', 'lightcoral', 'lightblue', 'lightcoral', 'lightblue']),
    line=dict(width=2, color='gray'),
    showlegend=False
))

# Añadir anotaciones para las transiciones
annotations = [
    dict(x=1.5, y=2.5, text='Observación<br>Estado', showarrow=True, ax=20, ay=-20),
    dict(x=2.5, y=2.5, text='Acción', showarrow=True, ax=-20, ay=-20),
    dict(x=2.5, y=1.5, text='Recompensa', showarrow=True, ax=-20, ay=20),
    dict(x=1.5, y=1.5, text='Nuevo Estado', showarrow=True, ax=20, ay=20)
]

fig.update_layout(
    title='Ciclo Percepción-Acción en Reinforcement Learning',
    xaxis=dict(showgrid=False, showticklabels=False, zeroline=False),
    yaxis=dict(showgrid=False, showticklabels=False, zeroline=False),
    annotations=annotations,
    template='plotly_white',
    height=500
)

fig.show()

print("\n📊 El agente:")
print("  1. Observa el estado del ambiente")
print("  2. Decide una acción basándose en su política")
print("  3. Ejecuta la acción")
print("  4. Recibe una recompensa y observa el nuevo estado")
print("  5. Repite el proceso (aprendiendo en el camino)")

### Ejemplo Visual: GridWorld

Vamos a crear un ambiente simple llamado **GridWorld** - una cuadrícula donde un agente debe navegar hacia un objetivo evitando obstáculos.

In [ ]:
class SimpleGridWorld:
    """
    Ambiente GridWorld simple para visualización.
    
    Simbología:
    - S: Start (inicio)
    - G: Goal (objetivo)
    - X: Obstáculo
    - .: Celda vacía
    """
    def __init__(self, size=5):
        self.size = size
        self.reset()
        
    def reset(self):
        """Reinicia el ambiente al estado inicial"""
        self.agent_pos = [0, 0]  # Posición del agente
        self.goal_pos = [4, 4]   # Posición del objetivo
        self.obstacles = [[1, 1], [2, 2], [3, 1]]  # Obstáculos
        return self.agent_pos
    
    def visualize(self):
        """Visualiza el estado actual del GridWorld"""
        grid = np.zeros((self.size, self.size))
        
        # Marcar obstáculos
        for obs in self.obstacles:
            grid[obs[0], obs[1]] = -1
        
        # Marcar objetivo
        grid[self.goal_pos[0], self.goal_pos[1]] = 2
        
        # Marcar agente
        grid[self.agent_pos[0], self.agent_pos[1]] = 1
        
        # Crear visualización
        fig = go.Figure(data=go.Heatmap(
            z=grid,
            colorscale=[
                [0, 'black'],      # Obstáculos
                [0.33, 'white'],   # Celdas vacías
                [0.66, 'lightblue'],  # Agente
                [1, 'lightgreen']  # Objetivo
            ],
            showscale=False
        ))
        
        # Añadir anotaciones
        annotations = []
        for i in range(self.size):
            for j in range(self.size):
                if [i, j] == self.agent_pos:
                    symbol = 'A'
                elif [i, j] == self.goal_pos:
                    symbol = 'G'
                elif [i, j] in self.obstacles:
                    symbol = 'X'
                else:
                    symbol = ''
                
                annotations.append(
                    dict(x=j, y=i, text=symbol, showarrow=False,
                         font=dict(size=20, color='darkblue'))
                )
        
        fig.update_layout(
            title='GridWorld: A=Agente, G=Objetivo, X=Obstáculo',
            xaxis=dict(showgrid=True, dtick=1),
            yaxis=dict(showgrid=True, dtick=1, autorange='reversed'),
            annotations=annotations,
            width=500,
            height=500
        )
        
        return fig

# Crear y visualizar GridWorld
env = SimpleGridWorld(size=5)
fig = env.visualize()
fig.show()

print("\n🎮 GridWorld creado!")
print(f"  - Agente en posición: {env.agent_pos}")
print(f"  - Objetivo en posición: {env.goal_pos}")
print(f"  - Obstáculos en: {env.obstacles}")

## 📐 Fundamentos Matemáticos

### Definición Formal de un Agente

Un **agente** es una entidad que:
1. Percibe su ambiente mediante **sensores**
2. Toma **acciones** mediante **actuadores**
3. Busca maximizar alguna noción de **recompensa** acumulada

#### Notación Fundamental

$$
\begin{align}
s_t &\in \mathcal{S} \tag{1} \\
\text{donde: } & \\
s_t &: \text{estado del ambiente en el tiempo } t \\
\mathcal{S} &: \text{conjunto de todos los estados posibles (espacio de estados)}
\end{align}
$$

$$
\begin{align}
a_t &\in \mathcal{A} \tag{2} \\
\text{donde: } & \\
a_t &: \text{acción tomada por el agente en el tiempo } t \\
\mathcal{A} &: \text{conjunto de todas las acciones posibles (espacio de acciones)}
\end{align}
$$

$$
\begin{align}
r_t &\in \mathbb{R} \tag{3} \\
\text{donde: } & \\
r_t &: \text{recompensa recibida en el tiempo } t
\end{align}
$$

### La Política

La **política** $\pi$ es la estrategia del agente - un mapeo de estados a acciones:

$$
\begin{align}
\pi: \mathcal{S} \rightarrow \mathcal{A} \tag{4}
\end{align}
$$

O en forma probabilística (política estocástica):

$$
\begin{align}
\pi(a|s) &= P(a_t = a \mid s_t = s) \tag{5} \\
\text{donde: } & \\
\pi(a|s) &: \text{probabilidad de tomar acción } a \text{ en estado } s
\end{align}
$$

### La Función de Recompensa

La función de recompensa define el objetivo del agente:

$$
\begin{align}
r_t &= R(s_t, a_t, s_{t+1}) \tag{6} \\
\text{donde: } & \\
R &: \mathcal{S} \times \mathcal{A} \times \mathcal{S} \rightarrow \mathbb{R}
\end{align}
$$

**Objetivo del agente:** Maximizar la recompensa acumulada (retorno)

$$
\begin{align}
G_t &= r_{t+1} + r_{t+2} + r_{t+3} + \cdots = \sum_{k=0}^{\infty} r_{t+k+1} \tag{7}
\end{align}
$$

### Ejemplo Numérico: GridWorld

Para nuestro GridWorld de 5×5:

- **Espacio de estados**: $\mathcal{S} = \{(i,j) : 0 \leq i,j < 5\}$ → 25 estados
- **Espacio de acciones**: $\mathcal{A} = \{\text{arriba}, \text{abajo}, \text{izquierda}, \text{derecha}\}$ → 4 acciones
- **Recompensas**:
  - $R(s, a, s_{\text{goal}}) = +10$ (llegar al objetivo)
  - $R(s, a, s_{\text{obstacle}}) = -5$ (chocar con obstáculo)
  - $R(s, a, s') = -1$ (cada paso, para incentivar eficiencia)

**Ejemplo de trayectoria**:

| Paso | Estado $(i,j)$ | Acción | Recompensa | Razón |
|------|---------------|--------|------------|-------|
| 0 | (0,0) | derecha | -1 | Movimiento normal |
| 1 | (0,1) | abajo | -1 | Movimiento normal |
| 2 | (1,1) | derecha | -5 | Intentó moverse a obstáculo |
| 3 | (1,1) | abajo | -1 | Movimiento normal |
| 4 | (2,1) | ... | ... | ... |

**Retorno total** para una trayectoria de 10 pasos que llega al objetivo:
$$G_0 = (-1) \times 9 + (+10) = 1$$

> 💡 **Insight**: El agente debe aprender a balancear eficiencia (menos pasos negativos) con seguridad (evitar obstáculos).

## 💻 Implementación Desde Cero

Ahora implementaremos un ambiente GridWorld completo y diferentes tipos de agentes.

In [ ]:
class GridWorldEnv:
    """
    Ambiente GridWorld completo para Reinforcement Learning.
    
    El agente debe navegar desde el inicio hasta el objetivo evitando obstáculos.
    
    Parámetros:
    -----------
    size : int
        Tamaño de la cuadrícula (size × size)
    goal_pos : tuple
        Posición (i, j) del objetivo
    obstacles : list
        Lista de posiciones (i, j) de obstáculos
    """
    
    def __init__(self, size: int = 5, 
                 goal_pos: Tuple[int, int] = (4, 4),
                 obstacles: List[Tuple[int, int]] = None):
        self.size = size
        self.goal_pos = goal_pos
        self.obstacles = obstacles if obstacles else [[1, 1], [2, 2], [3, 1]]
        
        # Espacios de estados y acciones
        self.state_space = [(i, j) for i in range(size) for j in range(size)]
        self.action_space = ['up', 'down', 'left', 'right']
        self.n_actions = len(self.action_space)
        
        # Mapeo de acciones a cambios en posición
        self.action_to_delta = {
            'up': (-1, 0),
            'down': (1, 0),
            'left': (0, -1),
            'right': (0, 1)
        }
        
        self.reset()
    
    def reset(self) -> Tuple[int, int]:
        """
        Reinicia el ambiente al estado inicial.
        
        Returns:
        --------
        state : tuple
            Estado inicial (posición del agente)
        """
        self.agent_pos = [0, 0]  # Siempre empezamos en la esquina superior izquierda
        self.done = False
        self.steps = 0
        return tuple(self.agent_pos)
    
    def step(self, action: str) -> Tuple[Tuple[int, int], float, bool, dict]:
        """
        Ejecuta una acción en el ambiente.
        
        Parameters:
        -----------
        action : str
            Acción a ejecutar ('up', 'down', 'left', 'right')
        
        Returns:
        --------
        next_state : tuple
            Nuevo estado después de la acción
        reward : float
            Recompensa recibida
        done : bool
            Si el episodio ha terminado
        info : dict
            Información adicional
        """
        if self.done:
            raise ValueError("El episodio ya terminó. Llama a reset() primero.")
        
        # Calcular nueva posición
        delta = self.action_to_delta[action]
        new_pos = [
            self.agent_pos[0] + delta[0],
            self.agent_pos[1] + delta[1]
        ]
        
        # Verificar si la nueva posición es válida
        reward = -1  # Penalización por cada paso (incentiva eficiencia)
        
        # Caso 1: Fuera de límites
        if (new_pos[0] < 0 or new_pos[0] >= self.size or 
            new_pos[1] < 0 or new_pos[1] >= self.size):
            # El agente se queda en su posición
            reward = -5  # Penalización mayor
        
        # Caso 2: Obstáculo
        elif new_pos in self.obstacles:
            # El agente se queda en su posición
            reward = -5  # Penalización mayor
        
        # Caso 3: Movimiento válido
        else:
            self.agent_pos = new_pos
            
            # Caso 3a: Llegó al objetivo
            if new_pos == list(self.goal_pos):
                reward = 10  # Recompensa grande por completar
                self.done = True
        
        self.steps += 1
        
        # Terminar si toma demasiados pasos (evitar loops infinitos)
        if self.steps >= 100:
            self.done = True
        
        info = {'steps': self.steps}
        
        return tuple(self.agent_pos), reward, self.done, info
    
    def render(self):
        """Imprime el estado actual del GridWorld en consola"""
        grid = [['.' for _ in range(self.size)] for _ in range(self.size)]
        
        # Marcar obstáculos
        for obs in self.obstacles:
            grid[obs[0]][obs[1]] = 'X'
        
        # Marcar objetivo
        grid[self.goal_pos[0]][self.goal_pos[1]] = 'G'
        
        # Marcar agente
        grid[self.agent_pos[0]][self.agent_pos[1]] = 'A'
        
        # Imprimir
        for row in grid:
            print(' '.join(row))
        print()

# Probar el ambiente
env = GridWorldEnv(size=5)
print("🌍 GridWorld Environment creado!\n")

print("Estado inicial:")
env.render()

print("Ejecutando acciones: derecha, derecha, abajo")
state, reward, done, info = env.step('right')
print(f"  Estado: {state}, Recompensa: {reward}")

state, reward, done, info = env.step('right')
print(f"  Estado: {state}, Recompensa: {reward}")

state, reward, done, info = env.step('down')
print(f"  Estado: {state}, Recompensa: {reward}\n")

print("Estado después de 3 acciones:")
env.render()

### Implementando Agentes

Ahora implementaremos diferentes tipos de agentes:

In [ ]:
class RandomAgent:
    """
    Agente que toma acciones aleatorias.
    
    Este es el agente más simple - no aprende nada, solo explora aleatoriamente.
    Útil como baseline para comparar otros agentes.
    """
    
    def __init__(self, action_space: List[str]):
        self.action_space = action_space
    
    def select_action(self, state: Tuple[int, int]) -> str:
        """
        Selecciona una acción aleatoria.
        
        Parameters:
        -----------
        state : tuple
            Estado actual (no usado por este agente)
        
        Returns:
        --------
        action : str
            Acción seleccionada aleatoriamente
        """
        return np.random.choice(self.action_space)


class GreedyAgent:
    """
    Agente que siempre se mueve hacia el objetivo (estrategia greedy).
    
    No aprende, pero usa conocimiento del objetivo para tomar decisiones.
    Problema: puede quedarse atascado con obstáculos.
    """
    
    def __init__(self, goal_pos: Tuple[int, int]):
        self.goal_pos = goal_pos
    
    def select_action(self, state: Tuple[int, int]) -> str:
        """
        Selecciona la acción que acerca más al objetivo.
        
        Parameters:
        -----------
        state : tuple
            Estado actual (posición del agente)
        
        Returns:
        --------
        action : str
            Acción que reduce la distancia al objetivo
        """
        i, j = state
        goal_i, goal_j = self.goal_pos
        
        # Decidir basándose en la diferencia con el objetivo
        if i < goal_i:
            return 'down'
        elif i > goal_i:
            return 'up'
        elif j < goal_j:
            return 'right'
        elif j > goal_j:
            return 'left'
        else:
            return 'down'  # Ya está en el objetivo


def run_episode(env: GridWorldEnv, agent, max_steps: int = 50) -> Tuple[float, int, List]:
    """
    Ejecuta un episodio completo con un agente.
    
    Parameters:
    -----------
    env : GridWorldEnv
        Ambiente
    agent : Agent
        Agente que interactúa con el ambiente
    max_steps : int
        Máximo número de pasos
    
    Returns:
    --------
    total_reward : float
        Recompensa total del episodio
    steps : int
        Número de pasos tomados
    trajectory : list
        Lista de (estado, acción, recompensa)
    """
    state = env.reset()
    total_reward = 0
    trajectory = []
    
    for step in range(max_steps):
        # Agente selecciona acción
        action = agent.select_action(state)
        
        # Ejecutar acción en el ambiente
        next_state, reward, done, info = env.step(action)
        
        # Registrar transición
        trajectory.append((state, action, reward))
        total_reward += reward
        
        if done:
            break
        
        state = next_state
    
    return total_reward, step + 1, trajectory


# Comparar agentes
print("🤖 Comparación de Agentes\n")
print("=" * 60)

env = GridWorldEnv(size=5)

# Agente aleatorio
random_agent = RandomAgent(env.action_space)
reward_random, steps_random, traj_random = run_episode(env, random_agent)
print(f"\n🎲 Agente Aleatorio:")
print(f"  Recompensa total: {reward_random}")
print(f"  Pasos tomados: {steps_random}")
print(f"  Llegó al objetivo: {env.done and env.agent_pos == list(env.goal_pos)}")

# Agente greedy
greedy_agent = GreedyAgent(env.goal_pos)
reward_greedy, steps_greedy, traj_greedy = run_episode(env, greedy_agent)
print(f"\n🎯 Agente Greedy:")
print(f"  Recompensa total: {reward_greedy}")
print(f"  Pasos tomados: {steps_greedy}")
print(f"  Llegó al objetivo: {env.done and env.agent_pos == list(env.goal_pos)}")

print("\n" + "=" * 60)
print("\n💡 Observación:")
print("  - El agente aleatorio explora pero es ineficiente")
print("  - El agente greedy es determinístico pero puede atascarse")
print("  - Necesitamos agentes que APRENDAN de la experiencia!")

### Visualización de Trayectorias

In [ ]:
def visualize_trajectory(env: GridWorldEnv, trajectory: List) -> go.Figure:
    """
    Visualiza la trayectoria de un agente en el GridWorld.
    """
    # Extraer posiciones de la trayectoria
    positions = [state for state, _, _ in trajectory]
    positions.append(env.agent_pos)  # Añadir posición final
    
    # Crear grid base
    grid = np.zeros((env.size, env.size))
    for obs in env.obstacles:
        grid[obs[0], obs[1]] = -1
    grid[env.goal_pos[0], env.goal_pos[1]] = 2
    
    fig = go.Figure()
    
    # Dibujar grid
    fig.add_trace(go.Heatmap(
        z=grid,
        colorscale=[
            [0, 'black'],
            [0.33, 'white'],
            [0.66, 'white'],
            [1, 'lightgreen']
        ],
        showscale=False,
        opacity=0.5
    ))
    
    # Dibujar trayectoria
    path_j = [pos[1] for pos in positions]
    path_i = [pos[0] for pos in positions]
    
    fig.add_trace(go.Scatter(
        x=path_j,
        y=path_i,
        mode='lines+markers',
        line=dict(color='blue', width=3),
        marker=dict(size=10, color='lightblue'),
        name='Trayectoria'
    ))
    
    # Marcar inicio y fin
    fig.add_trace(go.Scatter(
        x=[positions[0][1]],
        y=[positions[0][0]],
        mode='markers+text',
        marker=dict(size=15, color='red', symbol='star'),
        text=['START'],
        textposition='top center',
        name='Inicio'
    ))
    
    fig.update_layout(
        title='Trayectoria del Agente',
        xaxis=dict(showgrid=True, dtick=1),
        yaxis=dict(showgrid=True, dtick=1, autorange='reversed'),
        width=500,
        height=500
    )
    
    return fig

# Visualizar trayectoria del agente greedy
env_viz = GridWorldEnv(size=5)
greedy_agent_viz = GreedyAgent(env_viz.goal_pos)
_, _, trajectory_viz = run_episode(env_viz, greedy_agent_viz)

fig = visualize_trajectory(env_viz, trajectory_viz)
fig.show()

print(f"\n📊 Trayectoria visualizada: {len(trajectory_viz)} pasos")

## 🔧 Versión con Framework

Ahora veamos cómo usar **Gymnasium** (anteriormente OpenAI Gym), el framework estándar para RL.

In [ ]:
# Instalar gymnasium si no está disponible
try:
    import gymnasium as gym
    print("✅ Gymnasium ya está instalado")
except ImportError:
    print("📦 Instalando Gymnasium...")
    !pip install gymnasium -q
    import gymnasium as gym
    print("✅ Gymnasium instalado correctamente")

In [ ]:
# Usar un ambiente simple de Gymnasium: FrozenLake
print("🧊 FrozenLake Environment\n")
print("Descripción: El agente debe cruzar un lago congelado")
print("  S: Start (inicio)")
print("  F: Frozen (superficie congelada - segura)")
print("  H: Hole (agujero - termina episodio)")
print("  G: Goal (objetivo)\n")

# Crear ambiente
env = gym.make('FrozenLake-v1', desc=None, map_name="4x4", is_slippery=False, render_mode="ansi")

print("Ambiente creado:")
print(f"  Espacio de estados: {env.observation_space}")
print(f"  Espacio de acciones: {env.action_space}")
print(f"  Número de estados: {env.observation_space.n}")
print(f"  Número de acciones: {env.action_space.n}\n")

# Mapeo de acciones
action_names = {0: 'LEFT', 1: 'DOWN', 2: 'RIGHT', 3: 'UP'}

# Ejecutar episodio aleatorio
print("\n🎮 Ejecutando episodio con acciones aleatorias:\n")
state, info = env.reset(seed=42)

print("Estado inicial:")
print(env.render())

total_reward = 0
for step in range(10):
    action = env.action_space.sample()  # Acción aleatoria
    next_state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    
    print(f"\nPaso {step + 1}: Acción = {action_names[action]}")
    print(f"  Estado: {state} → {next_state}")
    print(f"  Recompensa: {reward}")
    print(env.render())
    
    total_reward += reward
    
    if done:
        print(f"\n{'🎉 ¡Objetivo alcanzado!' if reward > 0 else '💀 Cayó en un agujero'}")
        break
    
    state = next_state

print(f"\nRecompensa total: {total_reward}")
env.close()

print("\n" + "="*60)
print("\n💡 Observación sobre Gymnasium:")
print("  - Interfaz estandarizada para todos los ambientes")
print("  - API simple: reset(), step(), render(), close()")
print("  - Permite comparar algoritmos de forma justa")
print("  - Cientos de ambientes disponibles")

## 🎯 Ejercicios

### 🟢 Ejercicio 1: Modificar Recompensas

Modifica la función de recompensa del GridWorld para penalizar más fuertemente los choques con obstáculos.

In [ ]:
def ejercicio_1():
    """
    Objetivo: Comprender cómo las recompensas afectan el comportamiento del agente.
    
    Instrucciones:
    1. Crea un GridWorldEnv
    2. Modifica el código para que chocar con un obstáculo dé recompensa de -10
    3. Ejecuta 5 episodios con un agente aleatorio
    4. Retorna la recompensa promedio
    """
    # TODO: Tu código aquí
    pass

# Test
def test_ejercicio_1():
    resultado = ejercicio_1()
    assert resultado is not None, "❌ Pista: Debes retornar la recompensa promedio"
    assert isinstance(resultado, (int, float)), "❌ Pista: El resultado debe ser un número"
    print(f"✅ ¡Correcto! Recompensa promedio: {resultado:.2f}")
    print("   Las recompensas definen qué comportamientos son deseables.")
    return True

# test_ejercicio_1()  # Descomenta cuando completes el ejercicio

### 🟡 Ejercicio 2: Agente con Memoria

Implementa un agente que recuerde qué acciones fueron malas en el pasado.

In [ ]:
class MemoryAgent:
    """
    Agente que recuerda estados donde recibió recompensas negativas.
    
    Objetivo: Implementar un agente simple que aprende de experiencias malas.
    """
    def __init__(self, action_space: List[str]):
        self.action_space = action_space
        self.bad_states = set()  # Estados a evitar
    
    def select_action(self, state: Tuple[int, int]) -> str:
        """
        TODO: Implementa la selección de acción.
        
        Hint:
        - Si el estado está en bad_states, intenta una acción diferente
        - Si no, elige aleatoriamente
        """
        # TODO: Tu código aquí
        pass
    
    def update(self, state: Tuple[int, int], reward: float):
        """
        TODO: Actualiza la memoria con la experiencia.
        
        Hint:
        - Si la recompensa es muy negativa (< -2), marca el estado como malo
        """
        # TODO: Tu código aquí
        pass

def test_ejercicio_2():
    env = GridWorldEnv(size=5)
    agent = MemoryAgent(env.action_space)
    
    # Ejecutar episodio
    state = env.reset()
    for _ in range(20):
        action = agent.select_action(state)
        if action is None:
            print("❌ Pista: select_action() debe retornar una acción")
            return False
        
        next_state, reward, done, _ = env.step(action)
        agent.update(state, reward)
        
        if done:
            break
        state = next_state
    
    assert len(agent.bad_states) > 0, "❌ Pista: El agente debería recordar estados malos"
    print(f"✅ ¡Correcto! El agente recordó {len(agent.bad_states)} estados malos.")
    print("   Este es un ejemplo simple de aprendizaje de experiencia.")
    return True

# test_ejercicio_2()  # Descomenta cuando completes el ejercicio

### 🔴 Ejercicio 3: Análisis Comparativo de Políticas

Diseña un experimento para comparar diferentes políticas (aleatorio, greedy, memoria) en múltiples ambientes.

In [ ]:
def ejercicio_3_comparacion():
    """
    Objetivo: Realizar un análisis estadístico de diferentes agentes.
    
    Instrucciones:
    1. Crea 3 ambientes GridWorld con diferentes configuraciones de obstáculos
    2. Para cada ambiente, ejecuta 10 episodios con cada tipo de agente
    3. Calcula estadísticas: media, std de recompensas y pasos
    4. Crea una visualización comparativa con plotly
    5. Retorna un DataFrame con los resultados
    """
    import pandas as pd
    
    # TODO: Tu código aquí
    # Hint: Usa un diccionario para almacenar resultados
    # Hint: plotly.express.bar() es útil para comparaciones
    
    pass

def test_ejercicio_3():
    import pandas as pd
    
    resultado = ejercicio_3_comparacion()
    assert isinstance(resultado, pd.DataFrame), "❌ Pista: Debes retornar un DataFrame"
    assert len(resultado) > 0, "❌ Pista: El DataFrame no debe estar vacío"
    print("✅ ¡Excelente! Análisis comparativo completado.")
    print("   Has aplicado el método científico a RL.")
    return True

# test_ejercicio_3()  # Descomenta cuando completes el ejercicio

## 📚 Resumen

### Conceptos Clave

- **Agente**: Entidad que percibe y actúa en un ambiente para maximizar recompensas
- **Ambiente**: Entorno donde el agente opera, define estados, acciones y recompensas
- **Ciclo Percepción-Acción**: Observar → Decidir → Actuar → Recibir Recompensa → Repetir
- **Política (π)**: Estrategia del agente que mapea estados a acciones
- **Recompensa**: Señal escalar que indica qué tan buena fue una acción
- **Episodio**: Secuencia completa de interacciones desde el inicio hasta un estado terminal

### Diferencia con Otros Paradigmas

| Aspecto | Supervised Learning | Reinforcement Learning |
|---------|---------------------|------------------------|
| **Feedback** | Etiquetas correctas inmediatas | Recompensas diferidas |
| **Objetivo** | Minimizar error de predicción | Maximizar recompensa acumulada |
| **Datos** | Dataset fijo | Datos generados por interacción |
| **Exploración** | No aplica | Esencial para descubrir |

### Lo que viene

En este notebook vimos agentes simples que no aprenden realmente. En el siguiente notebook, introduciremos:
- **Procesos de Decisión de Markov (MDP)**: Formalización matemática completa
- **Ecuaciones de Bellman**: Base teórica del aprendizaje por refuerzo
- **Funciones de Valor**: Cómo evaluar qué tan buenos son los estados y acciones

## 🔗 Recursos Adicionales

### 📄 Papers Fundamentales

- **"Reinforcement Learning: An Introduction"** - Sutton & Barto (2018)
  - Capítulos 1-2 cubren exactamente estos conceptos
  - Disponible gratis: http://incompleteideas.net/book/the-book.html
  - Contexto: LA referencia definitiva en RL, escrita por dos pioneros

### 🎥 Videos Recomendados

- **"Introduction to Reinforcement Learning"** - David Silver (DeepMind)
  - Lecture 1: https://www.youtube.com/watch?v=2pWv7GOvuf0
  - Explicación clara del ciclo agente-ambiente

- **"Deep RL Bootcamp"** - Berkeley
  - https://sites.google.com/view/deep-rl-bootcamp/lectures
  - Lecture 1 cubre fundamentos

### 💻 Implementaciones de Referencia

- **Gymnasium (OpenAI Gym)**: https://gymnasium.farama.org/
  - Framework estándar para ambientes de RL
  - Documentación excelente y ejemplos

- **Spinning Up in Deep RL**: https://spinningup.openai.com/
  - Tutorial educativo de OpenAI
  - Implementaciones limpias de algoritmos

### 📖 Lecturas Complementarias

- **"A Brief Introduction to Reinforcement Learning"** - Mnih et al.
  - Artículo corto y accesible
  - Buen overview del campo

- **Blog "Lilian Weng - Introduction to RL"**
  - https://lilianweng.github.io/posts/2018-02-19-rl-overview/
  - Excelente visualizaciones y explicaciones

## ➡️ Próximo Paso

En el siguiente notebook aprenderás sobre **Procesos de Decisión de Markov (MDP)** y **Ecuaciones de Bellman**, que formalizan matemáticamente el problema del aprendizaje por refuerzo y proporcionan las herramientas para diseñar algoritmos óptimos.

**[Continuar con: 02. MDP y Ecuaciones de Bellman →](02-mdp-bellman.ipynb)**

---

<div align="center">
    
**¡Felicitaciones por completar el primer notebook de Reinforcement Learning! 🎉**

</div>